In [1]:
!python -m pip uninstall torch torchvision torchaudio -y
!python -m pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu130

Found existing installation: torch 2.14.1+cu130
Uninstalling torch-2.14.1+cu130:
  Successfully uninstalled torch-2.14.1+cu130
Found existing installation: torchvision 0.29.1+cu130
Uninstalling torchvision-0.29.1+cu130:
  Successfully uninstalled torchvision-0.29.1+cu130
Found existing installation: torchaudio 2.11.0+cu130
Uninstalling torchaudio-2.11.0+cu130:
  Successfully uninstalled torchaudio-2.11.0+cu130
Looking in indexes: https://download.pytorch.org/whl/cu130
  Using cached torch-2.14.1%2Bcu130-cp311-cp311-win_amd64.whl.metadata (37 kB)
  Using cached torchvision-0.29.1%2Bcu130-cp311-cp311-win_amd64.whl.metadata (5.8 kB)
  Using cached torchaudio-2.11.0%2Bcu130-cp311-cp311-win_amd64.whl.metadata (7.0 kB)
Using cached torch-2.14.1%2Bcu130-cp311-cp311-win_amd64.whl (1990.6 MB)
Using cached torchvision-0.29.1%2Bcu130-cp311-cp311-win_amd64.whl (6.6 MB)
Using cached torchaudio-2.11.0%2Bcu130-cp311-cp311-win_amd64.whl (1.7 MB)

   ---------------------------------------- 0/3 [torcha

In [2]:
pip install -U transformers accelerate pandas

Note: you may need to restart the kernel to use updated packages.


In [3]:
!nvidia-smi

Wed Nov 18 11:40:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 596.71                 Driver Version: 596.71         CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX 4000 Ada Gene...  WDDM  |   00000000:47:00.0  On |                  Off |
| 30%   48C    P8             10W /  130W |     666MiB /  20475MiB |     12%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    ), "GB")

    x = torch.rand(3, 3, device="cuda")
    print("GPU tensor test:", x)

PyTorch: 2.14.1+cu130
CUDA available: True
GPU: NVIDIA RTX 4000 Ada Generation
VRAM: 19.99 GB
GPU tensor test: tensor([[0.3164, 0.0599, 0.5732],
        [0.4369, 0.3948, 0.3276],
        [0.7504, 0.0273, 0.1662]], device='cuda:0')


In [5]:
import io
import zipfile
from pathlib import Path
import math, os, time
import numpy as np
from transformers import get_cosine_schedule_with_warmup
import hashlib, pickle, json

import pandas as pd
import torch
from transformers import AutoTokenizer, Mamba2ForCausalLM


c:\Users\HP\anaconda3\envs\phd_pytorch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
# !conda init
# !conda activate phd_pytorch

In [7]:
# MODEL_NAME = "AntonV/mamba2-130m-hf"

# ZIP_PATH = Path("Dataset") / "binary_multi_train.zip"

# OUTPUT_DIR = "./mamba2-130m-cpt-test"

# NUM_ROWS = 10
# MAX_LENGTH = 256
# EPOCHS = 1
# LEARNING_RATE = 5e-5

MODEL_NAME = "AntonV/mamba2-130m-hf"
DATASET_DIR = Path("Dataset")
OUTPUT_DIR = Path("./mamba2-130m-cpt-full")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NUM_ROWS = None        # None = all rows (use 500 for the pilot run)
MAX_STEPS = None       # None = full run (use 30 for the pilot run)

USE_FILENAME_KEY = True   # set from the diagnostic below
TOKENS_PATH = OUTPUT_DIR / f"tokens_{NUM_ROWS or 'all'}.bin"

TRAIN_CSVS = [
    ("binary_multi_train.zip", "binary_multi_train/CJPE_ext_SCI_HCs_Tribunals_daily_orders_multi.csv"),
]

BLOCK_SIZE = 1024
MICRO_BATCH = 4        # lower if you hit OOM
GRAD_ACCUM = 8
EPOCHS = 1
LEARNING_RATE = 5e-5
WARMUP_FRAC = 0.03
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0
# VAL_FRACTION = 0.02

MICRO_BATCH = 2
GRAD_ACCUM = 16

LOG_EVERY = 10
EVAL_EVERY = 500
SAVE_EVERY = 500
SEED = 42
USE_BF16 = True        # set False if you see NaNs/errors

torch.manual_seed(SEED)

In [8]:
def read_filenames(zip_name, csv_name):
    with zipfile.ZipFile(DATASET_DIR / zip_name) as z, z.open(csv_name) as f:
        return pd.read_csv(f, usecols=["filename"])["filename"].astype(str)

bt = read_filenames("binary_multi_train.zip", "binary_multi_train/CJPE_ext_SCI_HCs_Tribunals_daily_orders_multi.csv")
tt = read_filenames("ternary_multi_train.zip", "ternary_multi_train/CJPE_ext_SCI_HCs_tribunals_dailyorder_multi_wo_RoD_ternary.csv")
bte = read_filenames("binary_test.zip", "binary_test/CJPE_ext_SCI_HCs_Tribunals_daily_orders_test.csv")
tte = read_filenames("ternary_test.zip", "ternary_test/CJPE_ext_SCI_HCs_tribunals_dailyorder_test_wo_RoD_ternary.csv")

print("binary train: rows", len(bt), "| unique filenames", bt.nunique())
print("sample:", bt.head(5).tolist())
sb, st = set(bt), set(tt)
print("ternary-train filenames also in binary-train:", len(st & sb), "of", len(st))
print("binary-train  ∩ binary-test :", len(sb & set(bte)))
print("binary-train  ∩ ternary-test:", len(sb & set(tte)))
print("ternary-train ∩ binary-test :", len(st & set(bte)))

binary train: rows 702945 | unique filenames 702945
sample: ['SupremeCourt_2005_285', 'SupremeCourt_2003_1337', 'SupremeCourt_2017_580', 'SupremeCourt_1972_55', 'SupremeCourt_1950_40']
ternary-train filenames also in binary-train: 492090 of 493726
binary-train  ∩ binary-test : 114690
binary-train  ∩ ternary-test: 140557
ternary-train ∩ binary-test : 80514


In [9]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available. Check your NVIDIA driver "
        "and PyTorch CUDA installation."
    )

device = torch.device("cuda")

print("GPU:", torch.cuda.get_device_name(0))
print(
    "GPU memory:",
    round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
    "GB"
)

GPU: NVIDIA RTX 4000 Ada Generation
GPU memory: 19.99 GB


In [10]:
# if not ZIP_PATH.exists():
#     raise FileNotFoundError(f"Could not find: {ZIP_PATH.resolve()}")

# with zipfile.ZipFile(ZIP_PATH, "r") as archive:

#     csv_files = [
#         name for name in archive.namelist()
#         if name.lower().endswith(".csv")
#     ]

#     if not csv_files:
#         raise ValueError("No CSV files found inside the ZIP archive.")

#     print("\nCSV files found in ZIP:")
#     for name in csv_files:
#         print(name)

#     # Prefer the daily-orders variant if it exists.
#     preferred = [
#         name for name in csv_files
#         if "daily_orders" in name.lower()
#     ]

#     selected_csv = preferred[0] if preferred else csv_files[0]

#     print("\nSelected CSV:", selected_csv)

#     df = pd.read_csv(
#         io.BytesIO(archive.read(selected_csv))
#     )

# print("\nDataset columns:", df.columns.tolist())

# if "text" not in df.columns:
#     raise ValueError("The selected CSV does not contain a 'text' column.")

# # CPT uses only the legal text, not classification labels.
# texts = df["text"].dropna().astype(str).str.strip()
# texts = texts[texts != ""].drop_duplicates()
# if NUM_ROWS is not None:
#     texts = texts.head(NUM_ROWS)
# texts = texts.tolist()

# if len(texts) == 0:
#     raise ValueError("No usable text rows were found.")

# print(f"\nLoaded {len(texts):,} legal text rows.")

In [11]:
pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [12]:
import os
from dotenv import load_dotenv

load_dotenv()

HF_Token = os.getenv("HF_TOKEN")

if not HF_Token:
    raise ValueError("Hugging Face token not found")

print("Hugging face token loaded successfully")


Hugging face token loaded successfully


In [13]:
pip install sentencepiece tiktoken

Note: you may need to restart the kernel to use updated packages.


In [14]:
from transformers import AutoConfig
from huggingface_hub import HfApi

config = AutoConfig.from_pretrained(
    MODEL_NAME,
    token=HF_Token
)

print("Model vocabulary size:", config.vocab_size)
print("Hidden size:", config.hidden_size if hasattr(config, "hidden_size") else config.d_model)

# Inspect files available in the repository
api = HfApi(token=HF_Token)
files = api.list_repo_files(MODEL_NAME)

print("\nFiles in Mamba-2 repository:")
for file in files:
    print(file)

Model vocabulary size: 50288
Hidden size: 768

Files in Mamba-2 repository:
.gitattributes
README.md
config.json
model.safetensors
special_tokens_map.json
tokenizer.json
tokenizer_config.json


In [15]:
print("\nLoading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.model_max_length = int(1e9)  # silence the "sequence too long" warning

if tokenizer.eos_token_id is None:
    raise ValueError("Tokenizer does not have an EOS token.")

print("Tokenizer vocabulary size:", len(tokenizer))
print("EOS token:", tokenizer.eos_token)


Loading tokenizer...
Tokenizer vocabulary size: 50277
EOS token: <|endoftext|>


In [17]:
import hashlib, pickle, json

def doc_key(t):
    return hashlib.md5(" ".join(t.lower().split()).encode("utf-8")).digest()

def iter_chunks(zip_name, csv_name, chunksize=2000):
    with zipfile.ZipFile(DATASET_DIR / zip_name) as z, z.open(csv_name) as f:
        for chunk in pd.read_csv(f, usecols=["filename", "text"], chunksize=chunksize):
            chunk = chunk.dropna(subset=["text"])
            chunk["text"] = chunk["text"].astype(str).str.strip()
            chunk["filename"] = chunk["filename"].astype(str)
            yield chunk[chunk["text"] != ""]

# ---- 1) Collect every dev/test document so none of them enter CPT (cached) ----
EVAL_IDS_PATH = OUTPUT_DIR / "eval_ids.pkl"
if EVAL_IDS_PATH.exists():
    with open(EVAL_IDS_PATH, "rb") as f:
        eval_files, eval_hashes = pickle.load(f)
else:
    eval_files, eval_hashes = set(), set()
    for zp in sorted(DATASET_DIR.glob("*.zip")):
        if not any(k in zp.name.lower() for k in ("dev", "test")):
            continue
        with zipfile.ZipFile(zp) as z:
            names = [n for n in z.namelist() if n.lower().endswith(".csv")]
        for n in names:
            for chunk in iter_chunks(zp.name, n):
                eval_files.update(chunk["filename"])
                eval_hashes.update(doc_key(t) for t in chunk["text"])
            print(f"[eval] {zp.name}:{n} | files {len(eval_files):,} | docs {len(eval_hashes):,}")
    with open(EVAL_IDS_PATH, "wb") as f:
        pickle.dump((eval_files, eval_hashes), f)
print(f"Eval documents excluded from CPT: {len(eval_hashes):,}")

# ---- 2) Stream train CSVs -> dedup -> tokenize -> append to a uint16 file ----
tokenizer.model_max_length = int(1e9)
eos_id = tokenizer.eos_token_id
assert len(tokenizer) < 65536

if TOKENS_PATH.exists():
    print(f"{TOKENS_PATH.name} exists - skipping tokenization (delete it to rebuild).")
else:
    seen_files, seen_hashes = set(), set()
    n_docs = n_tokens = skipped_eval = skipped_dup = 0
    done = False
    tmp_path = TOKENS_PATH.with_suffix(".tmp")
    with open(tmp_path, "wb") as out:
        for zip_name, csv_name in TRAIN_CSVS:
            for chunk in iter_chunks(zip_name, csv_name):
                chunk_texts = chunk["text"].tolist()
                chunk_names = chunk["filename"].tolist()
                kept = []
                for t, nme in zip(chunk_texts, chunk_names):
                    k = doc_key(t)
                    if k in eval_hashes or (USE_FILENAME_KEY and nme in eval_files):
                        skipped_eval += 1
                        continue
                    if k in seen_hashes or (USE_FILENAME_KEY and nme in seen_files):
                        skipped_dup += 1
                        continue
                    seen_hashes.add(k)
                    seen_files.add(nme)
                    kept.append(t)
                if NUM_ROWS is not None:
                    kept = kept[: NUM_ROWS - n_docs]
                if kept:
                    enc = tokenizer(kept, add_special_tokens=False, truncation=False)["input_ids"]
                    arr = np.concatenate([np.asarray(ids + [eos_id], dtype=np.uint16) for ids in enc])
                    arr.tofile(out)
                    n_docs += len(kept)
                    n_tokens += len(arr)
                print(f"  docs {n_docs:,} | tokens {n_tokens:,} | skipped eval {skipped_eval:,} dup {skipped_dup:,}", end="\r")
                if NUM_ROWS is not None and n_docs >= NUM_ROWS:
                    done = True
                    break
            if done:
                break
    os.replace(tmp_path, TOKENS_PATH)
    stats = dict(unique_docs=n_docs, total_tokens=n_tokens,
                 skipped_in_dev_test=skipped_eval, skipped_duplicates=skipped_dup)
    TOKENS_PATH.with_suffix(".json").write_text(json.dumps(stats, indent=2))
    print("\n", stats)

# ---- 3) Memory-mapped blocks + train/val split ----
data = np.memmap(TOKENS_PATH, dtype=np.uint16, mode="r")
n_blocks = len(data) // BLOCK_SIZE
blocks = data[: n_blocks * BLOCK_SIZE].reshape(n_blocks, BLOCK_SIZE)

def get_blocks(ids):
    ids = np.sort(np.asarray(ids))
    return torch.from_numpy(blocks[ids].astype(np.int64))

# Validation = evenly spaced contiguous segments (avoids sharing documents with
# training blocks, and avoids a val set taken from just one court/year).
SEG = 16
n_seg = max(1, min(512, n_blocks // 20) // SEG)
starts = np.linspace(0, n_blocks - SEG, n_seg).astype(int)
val_np = np.unique(np.concatenate([np.arange(s, s + SEG) for s in starts]))
mask = np.ones(n_blocks, dtype=bool)
mask[val_np] = False
val_ids = torch.from_numpy(val_np)
train_ids = torch.from_numpy(np.flatnonzero(mask))
print(f"Blocks: {n_blocks:,} | train {len(train_ids):,} | val {len(val_ids):,} "
      f"| train tokens {len(train_ids) * BLOCK_SIZE:,}")

Eval documents excluded from CPT: 500,245
tokens_all.bin exists - skipping tokenization (delete it to rebuild).
Blocks: 288,113 | train 287,601 | val 512 | train tokens 294,503,424


In [18]:
print("\nLoading Mamba-2 130M model...")

model = Mamba2ForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32
)

model.to(device)
model.train()

print("Model loaded successfully.")
print("Parameters:", sum(p.numel() for p in model.parameters()))

# Disable inference caching during training.
model.config.use_cache = False


Loading Mamba-2 130M model...


Loading weights: 100%|██████████| 218/218 [00:00<00:00, 477.71it/s]


Model loaded successfully.
Parameters: 128989632


In [19]:
decay, no_decay = [], []
for p in model.parameters():
    if p.requires_grad:
        (decay if p.ndim >= 2 else no_decay).append(p)

optimizer = torch.optim.AdamW(
    [
        {"params": decay, "weight_decay": WEIGHT_DECAY},
        {"params": no_decay, "weight_decay": 0.0},
    ],
    lr=LEARNING_RATE,
    betas=(0.9, 0.95),
)

seqs_per_step = MICRO_BATCH * GRAD_ACCUM
steps_per_epoch = len(train_ids) // seqs_per_step
total_steps = steps_per_epoch * EPOCHS
if MAX_STEPS is not None:
    total_steps = min(total_steps, MAX_STEPS)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=max(1, int(WARMUP_FRAC * total_steps)),
    num_training_steps=total_steps,
)
print(f"Steps/epoch: {steps_per_epoch:,} | total steps: {total_steps:,} | "
      f"tokens/step: {seqs_per_step * BLOCK_SIZE:,}")

Steps/epoch: 8,987 | total steps: 8,987 | tokens/step: 32,768


In [20]:
autocast_ctx = lambda: torch.autocast(
    device_type="cuda", dtype=torch.bfloat16, enabled=USE_BF16
)

@torch.no_grad()
def evaluate():
    model.eval()
    losses = []
    for i in range(0, len(val_ids), MICRO_BATCH * 2):
        x = get_blocks(val_ids[i:i + MICRO_BATCH * 2]).to(device)
        with autocast_ctx():
            out = model(input_ids=x, labels=x, use_cache=False)
        losses.append(out.loss.item())
    model.train()
    loss = float(np.mean(losses))
    return loss, math.exp(loss)

CKPT_PATH = OUTPUT_DIR / "ckpt.pt"

def save_checkpoint(step):
    torch.save(
        {"model": model.state_dict(), "optimizer": optimizer.state_dict(),
         "scheduler": scheduler.state_dict(), "step": step},
        CKPT_PATH.with_suffix(".tmp"),
    )
    os.replace(CKPT_PATH.with_suffix(".tmp"), CKPT_PATH)

start_step = 0
if CKPT_PATH.exists():
    ckpt = torch.load(CKPT_PATH, map_location=device)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])
    start_step = ckpt["step"]
    print(f"Resumed from step {start_step}")
    del ckpt

if start_step == 0:
    v_loss, v_ppl = evaluate()
    print(f"Baseline (before CPT) | val loss {v_loss:.4f} | ppl {v_ppl:.2f}")

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `mamba_chunk_scan_combined` is falling back to its reference PyTorch implementation because `mamba_ssm` is not installed. This is correct but much slower; install `mamba_ssm` for the optimized kernel.


Baseline (before CPT) | val loss 3.3591 | ppl 28.76


In [21]:
print("\nStarting CPT...\n")
log_path = OUTPUT_DIR / "train_log.csv"
if start_step == 0:
    log_path.write_text("step,train_loss,lr,tokens_per_sec,val_loss,val_ppl\n")

torch.cuda.reset_peak_memory_stats()
t0 = time.time()
run_loss, run_steps, run_tokens = 0.0, 0, 0
t_log = time.time()

cached_epoch, order = -1, None

for step in range(start_step, total_steps):
    epoch, j = divmod(step, steps_per_epoch)
    if epoch != cached_epoch:   # shuffle once per epoch, not every step
        order = torch.randperm(len(train_ids), generator=torch.Generator().manual_seed(SEED + epoch))
        cached_epoch = epoch
    idx = order[j * seqs_per_step:(j + 1) * seqs_per_step]

    optimizer.zero_grad(set_to_none=True)
    step_loss = 0.0
    for m in range(GRAD_ACCUM):
        x = get_blocks(train_ids[idx[m * MICRO_BATCH:(m + 1) * MICRO_BATCH]]).to(device)
        with autocast_ctx():
            out = model(input_ids=x, labels=x, use_cache=False)
        loss = out.loss / GRAD_ACCUM
        loss.backward()
        step_loss += loss.item()

    if not math.isfinite(step_loss):
        raise RuntimeError(f"Non-finite loss at step {step}. Try USE_BF16=False or a lower LEARNING_RATE.")

    torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
    optimizer.step()
    scheduler.step()

    run_loss += step_loss
    run_steps += 1
    run_tokens += seqs_per_step * BLOCK_SIZE
    done = step + 1

    if done % LOG_EVERY == 0 or done == total_steps:
        now = time.time()
        tps = run_tokens / (now - t_log)
        eta_min = (total_steps - done) * seqs_per_step * BLOCK_SIZE / tps / 60
        lr = scheduler.get_last_lr()[0]
        avg = run_loss / run_steps
        print(f"step {done}/{total_steps} | loss {avg:.4f} | lr {lr:.2e} | "
              f"{tps:,.0f} tok/s | ETA {eta_min:,.1f} min | "
              f"peak mem {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")
        with open(log_path, "a") as f:
            f.write(f"{done},{avg:.5f},{lr:.3e},{tps:.0f},,\n")
        run_loss, run_steps, run_tokens, t_log = 0.0, 0, 0, now

    if done % EVAL_EVERY == 0 or done == total_steps:
        v_loss, v_ppl = evaluate()
        print(f"  >> val loss {v_loss:.4f} | ppl {v_ppl:.2f}")
        with open(log_path, "a") as f:
            f.write(f"{done},,,,{v_loss:.5f},{v_ppl:.3f}\n")

    if done % SAVE_EVERY == 0:
        save_checkpoint(done)

print(f"\nTraining finished in {(time.time() - t0) / 60:.1f} min")

[transformers] `mamba_split_conv1d_scan_combined` is falling back to its reference PyTorch implementation because `mamba_ssm` is not installed. This is correct but much slower; install `mamba_ssm` for the optimized kernel.



Starting CPT...

step 10/8987 | loss 3.3036 | lr 1.86e-06 | 4,585 tok/s | ETA 1,069.2 min | peak mem 11.11 GB
step 20/8987 | loss 3.3099 | lr 3.72e-06 | 4,600 tok/s | ETA 1,064.7 min | peak mem 11.11 GB
step 30/8987 | loss 3.2739 | lr 5.58e-06 | 4,537 tok/s | ETA 1,078.3 min | peak mem 11.11 GB
step 40/8987 | loss 3.2861 | lr 7.43e-06 | 4,535 tok/s | ETA 1,077.5 min | peak mem 11.11 GB
step 50/8987 | loss 3.2075 | lr 9.29e-06 | 4,503 tok/s | ETA 1,083.9 min | peak mem 11.11 GB
step 60/8987 | loss 3.2293 | lr 1.12e-05 | 4,515 tok/s | ETA 1,079.7 min | peak mem 11.11 GB
step 70/8987 | loss 3.1879 | lr 1.30e-05 | 4,594 tok/s | ETA 1,060.1 min | peak mem 11.11 GB
step 80/8987 | loss 3.1581 | lr 1.49e-05 | 4,593 tok/s | ETA 1,059.0 min | peak mem 11.11 GB
step 90/8987 | loss 3.1577 | lr 1.67e-05 | 4,586 tok/s | ETA 1,059.5 min | peak mem 11.11 GB
step 100/8987 | loss 3.1136 | lr 1.86e-05 | 4,587 tok/s | ETA 1,058.0 min | peak mem 11.11 GB
step 110/8987 | loss 3.0732 | lr 2.04e-05 | 4,594 t

In [22]:
final_dir = OUTPUT_DIR / "final"
final_dir.mkdir(exist_ok=True)

model.save_pretrained(final_dir)
tokenizer.save_pretrained(final_dir)

print("Saved to", final_dir.resolve())


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.76it/s]

Saved to C:\Users\HP\Desktop\2022IMT-061 Priya Keshri MTP\mamba2-130m-cpt-full\final


In [23]:
print(
    "\nPeak GPU memory allocated:",
    round(torch.cuda.max_memory_allocated() / 1024**3, 2),
    "GB"
)


Peak GPU memory allocated: 11.11 GB
